# 03 — Data Preprocessing
## Paddy Yield Predictor

Here we clean the data and prepare it for model training:
- Remove duplicates
- Split into features and target
- Split into train/test sets
- Build a preprocessing pipeline (scaling + encoding)

In [2]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.logger import get_logger
from src.data_loader import load_data, clean_data, split_features_target

log = get_logger('03_preprocessing')
log.info('Starting Preprocessing notebook')

2026-08-18 15:08:36 | INFO     | 03_preprocessing | Starting Preprocessing notebook


In [3]:
# Load and clean data
try:
    df = load_data(PROJECT_ROOT / 'paddydataset.csv')
    df = clean_data(df)
    print('Shape after cleaning:', df.shape)
except Exception as e:
    log.error(f'Failed during load/clean: {e}')
    raise

2026-08-18 15:08:37 | INFO     | src.data_loader | Dataset loaded — shape: (2789, 45)
2026-08-18 15:08:37 | INFO     | src.data_loader | Cleaning done — removed 451 duplicate/empty rows. Final shape: (2338, 45)


Shape after cleaning: (2338, 45)


In [4]:
# Split into features and target
try:
    X, y = split_features_target(df)
    print('X shape:', X.shape)
    print('y shape:', y.shape)
except Exception as e:
    log.error(f'Failed during feature/target split: {e}')
    raise

2026-08-18 15:08:37 | INFO     | src.data_loader | Features shape: (2338, 44) | Target shape: (2338,)


X shape: (2338, 44)
y shape: (2338,)


In [5]:
from sklearn.model_selection import train_test_split

try:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42
    )
    print('Training set size :', X_train.shape)
    print('Testing set size  :', X_test.shape)
    log.info(f'Train/test split done — Train: {X_train.shape}, Test: {X_test.shape}')
except Exception as e:
    log.error(f'Train/test split failed: {e}')
    raise

2026-08-18 15:08:41 | INFO     | 03_preprocessing | Train/test split done — Train: (1870, 44), Test: (468, 44)


Training set size : (1870, 44)
Testing set size  : (468, 44)


### Building the preprocessing pipeline

We need two different treatments for our columns:
- **Numeric columns** → fill missing values with median, then scale to same range
- **Categorical columns** → fill missing with most common value, then one-hot encode

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

try:
    num_cols = X.select_dtypes(include='number').columns.tolist()
    cat_cols = X.select_dtypes(exclude='number').columns.tolist()

    print('Numeric features  :', len(num_cols))
    print('Categorical features:', len(cat_cols))

    num_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    cat_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ])

    preprocessor = ColumnTransformer([
        ('num', num_pipeline, num_cols),
        ('cat', cat_pipeline, cat_cols)
    ])

    log.info('Preprocessor built successfully')
except Exception as e:
    log.error(f'Failed to build preprocessor: {e}')
    raise

2026-08-18 15:08:42 | INFO     | 03_preprocessing | Preprocessor built successfully


Numeric features  : 36
Categorical features: 8


In [7]:
# Apply the preprocessing to check it works
try:
    X_train_transformed = preprocessor.fit_transform(X_train)
    X_test_transformed  = preprocessor.transform(X_test)

    print('Transformed train shape:', X_train_transformed.shape)
    print('Transformed test shape :', X_test_transformed.shape)
    log.info('Preprocessing applied successfully')
except Exception as e:
    log.error(f'Preprocessing transform failed: {e}')
    raise

2026-08-18 15:08:42 | INFO     | 03_preprocessing | Preprocessing applied successfully


Transformed train shape: (1870, 71)
Transformed test shape : (468, 71)
